<a href="https://colab.research.google.com/github/enjy46/AWS-Terraform-DevOps/blob/main/malware_classifier3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing libraries and data set

In [ ]:
import kagglehub
import os
import pandas as pd

dataset_path = kagglehub.dataset_download('adilalizada/dataset-for-my-honours')

print("Dataset folder:", dataset_path)

file_path = os.path.join(dataset_path, "PE_Dataset_Labeled.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Using Colab cache for faster access to the 'dataset-for-my-honours' dataset.
Dataset folder: /kaggle/input/dataset-for-my-honours
Dataset shape: (34055, 55)


,Unnamed: 0,File_Name,Address_of_Entry_Point,Base_of_Code,Base_of_Data,Checksum,DLL_Characteristics,File_Alignment,Image_Base,Loader_Flags,...,e_oeminfo,e_ovno,e_res,e_res2,e_sp,e_ss,DLLs,Functions,Sections,Label
0,0,C:/Windows_PE_Files//07409496-a423-4a3e-b620-2...,0x0,0x1000,NaN,0x16379,0x160,0x1000,0x769ec66000000001,0x100000,...,0,0,7798812,7798824,184,0,NaN,NaN,"['', '', '', '']",Benign
1,1,C:/Windows_PE_Files//0ae3b998-9a38-4b72-a4c4-0...,0x0,0x1000,NaN,0xc47b,0x160,0x1000,0x769ec66000000001,0x100000,...,0,0,7798812,7798824,184,0,NaN,NaN,"['', '', '', '']",Benign
2,2,C:/Windows_PE_Files//0ae3b998-9a38-4b72-a4c4-0...,0x0,0x1000,NaN,0x143e4,0x160,0x1000,0x769ec66000000001,0x100000,...,0,0,7798812,7798824,184,0,NaN,NaN,"['', '', '', '']",Benign
3,3,C:/Windows_PE_Files//0ae3b998-9a38-4b72-a4c4-0...,0x0,0x1000,NaN,0xc47b,0x160,0x1000,0x769ec66000000001,0x100000,...,0,0,7798812,7798824,184,0,NaN,NaN,"['', '', '', '']",Benign
4,4,C:/Windows_PE_Files//1049fcc0b780db01e31600005...,0x37f0,0x1000,NaN,0x1e55d,0x4160,0x1000,0x769ec66000000001,0x100000,...,0,0,7798812,7798824,184,0,NaN,NaN,"['', '', '', '', '', '', '']",Benign


# Data Inspection

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34055 entries, 0 to 34054
Data columns (total 55 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Unnamed: 0                  34055 non-null  int64 
 1   File_Name                   34055 non-null  object
 2   Address_of_Entry_Point      34055 non-null  object
 3   Base_of_Code                34055 non-null  object
 4   Base_of_Data                19172 non-null  object
 5   Checksum                    34055 non-null  object
 6   DLL_Characteristics         34055 non-null  object
 7   File_Alignment              34055 non-null  object
 8   Image_Base                  34055 non-null  object
 9   Loader_Flags                34055 non-null  object
 10  Magic                       34055 non-null  object
 11  Major_Image_Version         34055 non-null  int64 
 12  Major_Linker_Version        34055 non-null  int64 
 13  Major_OS_Version            34055 non-null  in

## Explanation of the 55 features

### OPTIONAL HEADER FEATURES (Execution & Structure)

- **Address_of_Entry_Point** - Where execution starts. Malware uses unusual entry points to evade detection.
- **Base_of_Code** - Starting address of code section. Abnormal values indicate packing/obfuscation.
- **Base_of_Data** - Starting address of data section. Missing/abnormal values suggest manipulation.
- **Checksum** - File integrity value. Malware often has invalid or zero checksums.
- **DLL_Characteristics** - Flags controlling DLL behavior. Malware disables ASLR/DEP (0x0000).
- **File_Alignment** - How sections align in file. Unusual alignments indicate packing (e.g., UPX).
- **Image_Base** - Preferred load address. Malware uses unusual base addresses to avoid detection.
- **Loader_Flags** - Debugging flags. Usually zero; anomalies may indicate malware.
- **Magic** - 32-bit (0x10b) or 64-bit (0x20b). Mismatches indicate tampering.
- **Major_Image_Version** - File version number. Malware uses random versions.
- **Major_Linker_Version** - Linker version. Non-standard versions suggest custom/bootleg compilers.
- **Major_OS_Version** - Minimum OS required. Malware targets old/vulnerable OS versions.
- **Major_Subsystem_Version** - Subsystem version (GUI/Console/Driver). Malware uses unusual combinations.
- **Minor_Image_Version** - Minor version number. Similar to Major_Image_Version.
- **Minor_Linker_Version** - Minor linker version. Unusual values indicate malware.
- **Minor_OS_Version** - Minor OS version. Anomalies suggest targeting specific vulnerabilities.
- **Minor_Subsystem_Version** - Minor subsystem version. Suspicious values indicate tampering.
- **Number_of_Rva_and_Sizes** - Number of data directories. Malware has fewer directories (packed).
- **Section_Alignment** - Memory alignment of sections. Abnormal values suggest packing.
- **Size_of_Code** - Size of executable code. Tiny size = packed/encrypted malware.
- **Size_of_Headers** - Size of PE headers. Unusual sizes indicate manipulation.
- **Size_of_Heap_Commit** - Heap memory to commit. Abnormal values suggest malicious behavior.
- **Size_of_Heap_Reserve** - Heap memory to reserve. Similar to above.
- **Size_of_Image** - Total file size in memory. Discrepancies indicate packing.
- **Size_of_Initialized_Data** - Size of initialized data. Malware hides payloads here.
- **Size_of_Stack_Commit** - Stack memory to commit. Abnormal values suggest exploits.
- **Size_of_Stack_Reserve** - Stack memory to reserve. Similar to above.
- **Size_of_Uninitialized_Data** - Size of uninitialized data (.bss). Unusual values indicate malware.
- **Subsystem** - GUI (2), Console (3), Driver (1). Malware uses incorrect subsystem values.
- **Win32_Version_Value** - Windows version info. Often zero; anomalies may indicate malware.

---

### DOS HEADER FEATURES

- **e_cblp** - Bytes on last page of file. Usually zero; anomalies indicate tampering.
- **e_cp** - Pages in file. Usually zero; anomalies indicate tampering.
- **e_cparhdr** - Size of header in paragraphs. Usually zero; anomalies indicate tampering.
- **e_crlc** - Relocation entries. Usually zero; anomalies indicate tampering.
- **e_cs** - Code segment. Usually zero; anomalies indicate tampering.
- **e_csum** - Checksum. Usually zero; anomalies indicate tampering.
- **e_ip** - Initial IP value. Usually zero; anomalies indicate tampering.
- **e_lfanew** Offset to PE header. Malware uses abnormal offsets to hide PE header.
- **e_lfarlc** - Relocation table offset. Usually zero; anomalies indicate tampering.
- **e_magic** - Magic number (0x5A4D = "MZ"). Should always be 0x5A4D; anomalies = corruption.
- **e_maxalloc** - Maximum extra memory. Usually zero; anomalies indicate tampering.
- **e_minalloc** - Minimum extra memory. Usually zero; anomalies indicate tampering.
- **e_oemid** - OEM identifier. Usually zero; anomalies indicate tampering.
- **e_oeminfo** - OEM information. Usually zero; anomalies indicate tampering.
- **e_ovno** - Overlay number. Usually zero; anomalies indicate tampering.
- **e_res** -  Reserved field #1. Malware hides payloads/keys here.
- **e_res2** - Reserved field #2.Should be zero. Malware hides payloads/keys here.
- **e_sp** - Initial SP value. Usually zero; anomalies indicate tampering.
- **e_ss** - Initial SS value. Usually zero; anomalies indicate tampering.

---

### TEXT FEATURES (Converted to Numerical Counts)

- **DLLs** - List of imported DLLs (e.g., kernel32.dll, user32.dll). Malware imports suspicious DLLs. Converted to DLL_count.
- **Functions** - List of imported API functions (e.g., CreateRemoteThread, WriteProcessMemory). Malware uses suspicious APIs. Converted to **Function_count**.
- **Sections** - List of PE section names (e.g., .text, .data, .rsrc). Malware has unusual sections (e.g., .UPX, .aspack). Converted to **Section_count**.

---

### TARGET VARIABLE

- **Label** - Classification: "Benign" or "Malicious". Converted to binary: 0 = Benign, 1 = Malware.

In [ ]:
print("Missing values:", df.isnull().sum().sum())

Missing values: 45819


In [ ]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [ ]:
print(df.columns)

Index(['Unnamed: 0', 'File_Name', 'Address_of_Entry_Point', 'Base_of_Code',
       'Base_of_Data', 'Checksum', 'DLL_Characteristics', 'File_Alignment',
       'Image_Base', 'Loader_Flags', 'Magic', 'Major_Image_Version',
       'Major_Linker_Version', 'Major_OS_Version', 'Major_Subsystem_Version',
       'Minor_Image_Version', 'Minor_Linker_Version', 'Minor_OS_Version',
       'Minor_Subsystem_Version', 'Number_of_Rva_and_Sizes',
       'Section_Alignment', 'Size_of_Code', 'Size_of_Headers',
       'Size_of_Heap_Commit', 'Size_of_Heap_Reserve', 'Size_of_Image',
       'Size_of_Initialized_Data', 'Size_of_Stack_Commit',
       'Size_of_Stack_Reserve', 'Size_of_Uninitialized_Data', 'Subsystem',
       'Win32_Version_Value', 'e_cblp', 'e_cp', 'e_cparhdr', 'e_crlc', 'e_cs',
       'e_csum', 'e_ip', 'e_lfanew', 'e_lfarlc', 'e_magic', 'e_maxalloc',
       'e_minalloc', 'e_oemid', 'e_oeminfo', 'e_ovno', 'e_res', 'e_res2',
       'e_sp', 'e_ss', 'DLLs', 'Functions', 'Sections', 'Label'],
     

# Data Cleaning

In [ ]:
missing_per_col = df.isnull().sum().sort_values(ascending=False)
print(missing_per_col[missing_per_col > 0])

DLLs            15578
Functions       15358
Base_of_Data    14883
dtype: int64


In [ ]:
print(missing_per_col[missing_per_col > 0])

DLLs            15578
Functions       15358
Base_of_Data    14883
dtype: int64


In [ ]:
# Create a working copy before cleaning
df_clean = df.copy()

# Feature engineering from DLLs, Functions, and Sections
# These columns are text-like, so instead of keeping them raw,
# we convert them into simple numerical features that the model can use.

def count_list_items(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()
    if x == "" or x == "[]" or x == "['']":
        return 0

    try:
        import ast
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            cleaned = [item for item in parsed if str(item).strip() != ""]
            return len(cleaned)
        return 0
    except:
        return 0

df_clean["DLL_count"] = df_clean["DLLs"].apply(count_list_items)
df_clean["Function_count"] = df_clean["Functions"].apply(count_list_items)
df_clean["Section_count"] = df_clean["Sections"].apply(count_list_items)

### Dropped complex columns that may mess the model

In [ ]:

# Drop raw text-heavy columns after extracting useful counts from them
df_clean = df_clean.drop(columns=["DLLs", "Functions", "Sections"])

In [ ]:
df_clean["Base_of_Data"] = df_clean["Base_of_Data"].fillna("0x0")

### Convert hexadecimal features to numerical values

Many features are stored as hexadecimal strings like 0x1000 which must be converted to integers for machine learning.

In [ ]:
def hex_to_int(val):
    if pd.isna(val):
        return val
    if isinstance(val, str) and val.startswith("0x"):
        return int(val, 16)
    return val

for col in df_clean.columns:
    if col not in ["Label", "File_Name"]:
        df_clean[col] = df_clean[col].apply(hex_to_int)

### Filling missing values

In [ ]:
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

In [ ]:
print("Missing values after cleaning:", df_clean.isnull().sum().sum())
df_clean.info()

Missing values after cleaning: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34055 entries, 0 to 34054
Data columns (total 55 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Unnamed: 0                  34055 non-null  int64 
 1   File_Name                   34055 non-null  object
 2   Address_of_Entry_Point      34055 non-null  int64 
 3   Base_of_Code                34055 non-null  int64 
 4   Base_of_Data                34055 non-null  int64 
 5   Checksum                    34055 non-null  int64 
 6   DLL_Characteristics         34055 non-null  int64 
 7   File_Alignment              34055 non-null  int64 
 8   Image_Base                  34055 non-null  int64 
 9   Loader_Flags                34055 non-null  int64 
 10  Magic                       34055 non-null  int64 
 11  Major_Image_Version         34055 non-null  int64 
 12  Major_Linker_Version        34055 non-null  int64 
 13  Major_OS_Vers

In [ ]:
df_clean = df_clean.drop(columns=[
    "Unnamed: 0",
    "File_Name",
    "Checksum",
    "Loader_Flags",
    "Win32_Version_Value"
])

In [ ]:
print("Missing values after cleaning:", df_clean.isnull().sum().sum())
print("Duplicate rows after cleaning:", df_clean.duplicated().sum())
df_clean.info()
df_clean.head()

Missing values after cleaning: 0
Duplicate rows after cleaning: 18824
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34055 entries, 0 to 34054
Data columns (total 50 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Address_of_Entry_Point      34055 non-null  int64
 1   Base_of_Code                34055 non-null  int64
 2   Base_of_Data                34055 non-null  int64
 3   DLL_Characteristics         34055 non-null  int64
 4   File_Alignment              34055 non-null  int64
 5   Image_Base                  34055 non-null  int64
 6   Magic                       34055 non-null  int64
 7   Major_Image_Version         34055 non-null  int64
 8   Major_Linker_Version        34055 non-null  int64
 9   Major_OS_Version            34055 non-null  int64
 10  Major_Subsystem_Version     34055 non-null  int64
 11  Minor_Image_Version         34055 non-null  int64
 12  Minor_Linker_Version        34055 non-null  in

,Address_of_Entry_Point,Base_of_Code,Base_of_Data,DLL_Characteristics,File_Alignment,Image_Base,Magic,Major_Image_Version,Major_Linker_Version,Major_OS_Version,...,e_oeminfo,e_ovno,e_res,e_res2,e_sp,e_ss,Label,DLL_count,Function_count,Section_count
0,0,4096,0,352,4096,8547487258414940161,523,10,14,10,...,0,0,7798812,7798824,184,0,0,0,0,0
1,0,4096,0,352,4096,8547487258414940161,523,10,14,10,...,0,0,7798812,7798824,184,0,0,0,0,0
2,0,4096,0,352,4096,8547487258414940161,523,10,14,10,...,0,0,7798812,7798824,184,0,0,0,0,0
3,0,4096,0,352,4096,8547487258414940161,523,10,14,10,...,0,0,7798812,7798824,184,0,0,0,0,0
4,14320,4096,0,16736,4096,8547487258414940161,523,10,14,10,...,0,0,7798812,7798824,184,0,0,0,0,0


In [ ]:
df_clean = df_clean.drop_duplicates()

print("Shape after dropping duplicates:", df_clean.shape)
print("Remaining duplicates:", df_clean.duplicated().sum())
print(df_clean["Label"].value_counts())

Shape after dropping duplicates: (15231, 50)
Remaining duplicates: 0
Label
0    10423
1     4808
Name: count, dtype: int64


# Data Preparation for Modelling

### Encode the target label

The target variable is converted from text labels into binary values.


In [ ]:
print(df_clean["Label"].unique())

df_clean["Label"] = df_clean["Label"].astype(str).str.strip().str.lower()

df_clean["Label"] = df_clean["Label"].map({
    "benign": 0,
    "malicious": 1,
    "malware": 1
})

print(df_clean["Label"].value_counts(dropna=False))

['Benign' 'Malicious']
Label
0    18627
1    15428
Name: count, dtype: int64


# Modelling

### Seperate fetures and target
* Separate features (X) and target variable (y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_full = df_clean.drop(columns=["Label"])
y = df_clean["Label"]

rf_temp = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_temp.fit(X_full, y)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

### Feature selection

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": X_full.columns,
    "Importance": rf_temp.feature_importances_
}).sort_values(by="Importance", ascending=False)

selected_features = importance_df.head(15)["Feature"].tolist()

print("Selected Features:", selected_features)

Selected Features: ['Minor_Linker_Version', 'Major_Image_Version', 'Major_OS_Version', 'Major_Subsystem_Version', 'DLL_Characteristics', 'Major_Linker_Version', 'e_lfanew', 'e_res2', 'e_res', 'Subsystem', 'Section_count', 'Number_of_Rva_and_Sizes', 'Size_of_Code', 'Magic', 'Size_of_Initialized_Data']


In [ ]:
X = df_clean[selected_features]
y = df_clean["Label"]

print("X shape:", X.shape)
print("Selected columns:", X.columns.tolist())

X shape: (15231, 15)
Selected columns: ['Minor_Linker_Version', 'Major_Image_Version', 'Major_OS_Version', 'Major_Subsystem_Version', 'DLL_Characteristics', 'Major_Linker_Version', 'e_lfanew', 'e_res2', 'e_res', 'Subsystem', 'Section_count', 'Number_of_Rva_and_Sizes', 'Size_of_Code', 'Magic', 'Size_of_Initialized_Data']


###Train/Test Split


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("\nTraining label distribution:")
print(y_train.value_counts())
print("\nTest label distribution:")
print(y_test.value_counts())

Training samples: 12184
Test samples: 3047

Training label distribution:
Label
0    8338
1    3846
Name: count, dtype: int64

Test label distribution:
Label
0    2085
1     962
Name: count, dtype: int64


### Training Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

start = time.time()

dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

dt_model.fit(X_train, y_train)

dt_time = time.time() - start

y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

print("Decision Tree trained.")
print("Training time:", dt_time)
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

Decision Tree trained.
Training time: 0.03896474838256836
Accuracy: 0.9894978667541845
F1 Score: 0.9833679833679834
ROC-AUC: 0.9880828808886363

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2085
           1       0.98      0.98      0.98       962

    accuracy                           0.99      3047
   macro avg       0.99      0.99      0.99      3047
weighted avg       0.99      0.99      0.99      3047


Confusion Matrix:
[[2069   16]
 [  16  946]]


### Training Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

rf_time = time.time() - start

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest trained.")
print("Training time:", rf_time)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest trained.
Training time: 0.5862808227539062
Accuracy: 0.9927797833935018
F1 Score: 0.9886363636363636
ROC-AUC: 0.9995802110910024

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2085
           1       0.98      0.99      0.99       962

    accuracy                           0.99      3047
   macro avg       0.99      0.99      0.99      3047
weighted avg       0.99      0.99      0.99      3047


Confusion Matrix:
[[2068   17]
 [   5  957]]


### Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_df.head(20))

                     Feature  Importance
0       Minor_Linker_Version    0.220266
2           Major_OS_Version    0.148938
1        Major_Image_Version    0.138159
3    Major_Subsystem_Version    0.114632
5       Major_Linker_Version    0.081581
4        DLL_Characteristics    0.068604
8                      e_res    0.040683
7                     e_res2    0.037037
9                  Subsystem    0.032009
12              Size_of_Code    0.030842
6                   e_lfanew    0.020455
11   Number_of_Rva_and_Sizes    0.019730
14  Size_of_Initialized_Data    0.019368
10             Section_count    0.015224
13                     Magic    0.012472


### Training SVM

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

start = time.time()

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

svm_time = time.time() - start

y_pred_svm = svm_model.predict(X_test)
y_prob_svm = svm_model.predict_proba(X_test)[:, 1]

print("SVM trained.")
print("Training time:", svm_time)
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("F1 Score:", f1_score(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_svm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

SVM trained.
Training time: 4.523700475692749
Accuracy: 0.9783393501805054
F1 Score: 0.965979381443299
ROC-AUC: 0.996706501742473

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      2085
           1       0.96      0.97      0.97       962

    accuracy                           0.98      3047
   macro avg       0.97      0.98      0.98      3047
weighted avg       0.98      0.98      0.98      3047


Confusion Matrix:
[[2044   41]
 [  25  937]]


### evaluation of the 3 models (comparison)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandas as pd

results = [
    {
        "Model": "Decision Tree",
        "Accuracy": accuracy_score(y_test, y_pred_dt),
        "F1 Score": f1_score(y_test, y_pred_dt),
        "ROC-AUC": roc_auc_score(y_test, y_prob_dt),
        "Training Time (s)": dt_time
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "F1 Score": f1_score(y_test, y_pred_rf),
        "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
        "Training Time (s)": rf_time
    },
    {
        "Model": "SVM",
        "Accuracy": accuracy_score(y_test, y_pred_svm),
        "F1 Score": f1_score(y_test, y_pred_svm),
        "ROC-AUC": roc_auc_score(y_test, y_prob_svm),
        "Training Time (s)": svm_time
    }
]

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False)

print(results_df)


# cross validate
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_rf = cross_validate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),
    X, y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("RF CV Accuracy:", cv_rf["test_accuracy"].mean())
print("RF CV F1:", cv_rf["test_f1"].mean())
print("RF CV ROC-AUC:", cv_rf["test_roc_auc"].mean())

           Model  Accuracy  F1 Score   ROC-AUC  Training Time (s)
1  Random Forest  0.992780  0.988636  0.999580           0.586281
0  Decision Tree  0.989498  0.983368  0.988083           0.038965
2            SVM  0.978339  0.965979  0.996707           4.523700
RF CV Accuracy: 0.9938284236391951
RF CV F1: 0.9902518138796849
RF CV ROC-AUC: 0.9994689715509804


### Random Forest performed best across all metrics.

### Download the model

In [ ]:
import joblib

joblib.dump(
    {
        "model": rf_model,
        "features": X.columns.tolist()
    },
    "malware_detector_model.pkl"
)

from google.colab import files

files.download("malware_detector_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>